# Loans at Risk: Capturing Default — Modeling

## Purpose

This notebook implements the modeling stage of the *Loans at Risk: Capturing Default* project.

The analysis uses the `feature_base` datasets produced in the ETL pipeline:

- **Training set:** loans issued between 2007 and 2014  
- **Testing set:** loans issued in 2015  

The modeling population is restricted to loans with terminal outcomes in order to define a clear prediction target.

The purpose of this notebook is to train and evaluate a small set of representative predictive models on this population and identify the model that produces the most accurate predictions of borrower default risk.

---

## Analytical Approach

Default prediction is formulated as a binary classification problem.

Loans are classified as:

- **Default (1):** Charged Off or Default  
- **Non-default (0):** Fully Paid  

Two policy-related loan status categories are normalized before modeling:

- “Does not meet the credit policy. Status: Fully Paid” → Fully Paid  
- “Does not meet the credit policy. Status: Charged Off” → Charged Off  

This preserves the economic outcome of the loan while removing administrative distinctions that are not relevant to the prediction task.

Model performance is evaluated primarily using **ROC-AUC**, with additional metrics including precision, recall, F1 score, and confusion matrices. Performance is compared between the training and testing datasets to assess predictive accuracy and potential overfitting.

---

## Model Strategy

Three models are evaluated in this analysis:

1. **Logistic Regression**  
2. **Random Forest**  
3. **CatBoost**

These models represent three different approaches to predictive modeling: a classical statistical model, a bagging-based tree ensemble, and a boosting-based ensemble method.

### Logistic Regression

Logistic Regression serves as the baseline model and represents the traditional statistical approach widely used in credit risk modeling (Thomas et al., 2017).

The model combines borrower characteristics into a weighted linear score. Each feature contributes positively or negatively depending on how strongly it is associated with default risk. Because this score can take any value, it is transformed using the **logistic function**, which maps the score onto a value between 0 and 1. The result can therefore be interpreted as the predicted probability that a loan will default. Logistic regression has historically been the dominant methodology used in credit scoring because it produces stable probability estimates and remains relatively interpretable compared to more complex machine learning models. It therefore provides a natural benchmark against which more flexible nonlinear models can be evaluated.

### Random Forest

Random Forest represents the **bagging ensemble paradigm** introduced by Breiman (2001).

A decision tree predicts outcomes by repeatedly dividing the data into smaller groups based on feature values. Each split attempts to separate loans with different outcomes—for example, separating borrowers with high debt ratios from those with lower ones. A single decision tree can be unstable because small changes in the data may lead to very different splits. Random Forest addresses this by building **many trees**, each trained on a slightly different random sample of the data and predictor variables. Each tree produces its own prediction, and the final model output is obtained by averaging the predictions across all trees. Combining many trees in this way reduces the instability of individual trees while allowing the model to capture nonlinear relationships and interactions between variables.

### CatBoost

CatBoost represents the **boosting ensemble paradigm**, where models are trained sequentially rather than independently (Prokhorenkova et al., 2018).

Like Random Forest, CatBoost uses decision trees as its building blocks. However, instead of training many independent trees, boosting trains trees **one after another**. Each new tree focuses on correcting prediction errors made by the previous trees. Over many iterations the model gradually improves its predictions by concentrating on the observations that were most difficult to predict earlier in the training process. CatBoost also includes techniques designed for tabular datasets containing categorical variables and aims to reduce bias during model training. Empirical research shows that tree-based ensembles remain effective approaches for tabular prediction tasks (Grinsztajn et al., 2022).

---

Together these models represent three distinct learning approaches:

- linear statistical modeling  
- bagged tree ensembles  
- boosted tree ensembles  

Their performance is compared in order to identify the model that produces the most accurate predictions of borrower default risk.

---

## Structure

The notebook proceeds in four stages.

1. **Modeling Population**  
   The modeling population is finalized and the binary target variable is constructed.

2. **Feature Engineering**  
   Additional feature transformations are performed to prepare the dataset for model training.

3. **Model Training**  
   Logistic Regression, Random Forest, and CatBoost models are trained on the training dataset.

4. **Model Evaluation**  
   Model performance is evaluated on both the training and testing datasets in order to identify the best-performing model.

---

In [1]:
from pathlib import Path
import sys

current_path = Path.cwd().resolve()
project_root = None

for parent_path in (current_path, *current_path.parents):
    if (parent_path / "pyproject.toml").exists():
        project_root = parent_path
        break

if project_root is None:
    raise RuntimeError(
        f"Failed to resolve project root: pyproject.toml not found from {current_path}"
    )

src_path = project_root / "src"
if not src_path.exists():
    raise RuntimeError(f"Expected 'src' directory at: {src_path}")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

{
    "stage": "bootstrap_import_path_ready",
    "project_root": str(project_root),
}

{'stage': 'bootstrap_import_path_ready',
 'project_root': 'D:\\Portfolio\\loans-at-risk-capturing-default'}

In [2]:
# ===========================================
# Imports: libraries and project modules
# ===========================================

from datetime import datetime, timezone
from typing import Callable
import uuid

from IPython.display import display
import pandas as pd

import config.logging as log_config
import analysis.dataset_artifacts as da
import analysis.modeling_artifacts as ma
import plots.report_figures as rf
import features.modeling_population as mp
import features.feature_utils as fu
import features.feature_engineering as fe
import modeling.train_models as tm
import modeling.evaluate_models as em


In [3]:
# ===============================
# Paths and run context
# ===============================

logs_root = project_root / "logs"
logs_root.mkdir(parents=True, exist_ok=True)

PROJECT_LOG_FILE = logs_root / "project.log"

run_id = uuid.uuid4().hex[:10]
run_timestamp_utc = datetime.now(timezone.utc)

run_header = (
    "NEW RUN: "
    f"{run_timestamp_utc.strftime('%Y-%m-%d %H:%M:%S')} UTC | "
    f"RUN_ID={run_id}"
)

log_config.log_messages("\n" + "=" * 60, PROJECT_LOG_FILE)
log_config.log_messages(run_header, PROJECT_LOG_FILE)
log_config.log_messages("=" * 60 + "\n", PROJECT_LOG_FILE)

log: Callable[[str], None] = log_config.get_logger(PROJECT_LOG_FILE)
log("Modeling notebook initialized")
log(f"project_root: {project_root}")
log(run_header)

{
    "stage": "run_started",
    "run_id": run_id,
    "utc_timestamp": run_timestamp_utc.isoformat(),
}

# ---------------------------------------------------------------
# Inputs for this notebook (processed, report)
# ---------------------------------------------------------------

feature_base_train_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2007_2014.parquet"
feature_base_test_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2015.parquet"

shared_model_train_data_file = project_root / "data" / "processed" / "model_train_data.parquet"
shared_model_test_data_file = project_root / "data" / "processed" / "model_test_data.parquet"

tree_no_log_train_data_file = project_root / "data" / "processed" / "tree_no_log_train_data.parquet"
tree_no_log_test_data_file = project_root / "data" / "processed" / "tree_no_log_test_data.parquet"

catboost_native_train_data_file = project_root / "data" / "processed" / "catboost_native_train_data.parquet"
catboost_native_test_data_file = project_root / "data" / "processed" / "catboost_native_test_data.parquet"

artifacts_dir = project_root / "artifacts"
modeling_artifacts_dir = artifacts_dir / "modeling"
modeling_tables_dir = modeling_artifacts_dir / "tables"
modeling_figures_dir = modeling_artifacts_dir / "figures"

modeling_tables_dir.mkdir(parents=True, exist_ok=True)
modeling_figures_dir.mkdir(parents=True, exist_ok=True)

log(f"Shared model train parquet path: {shared_model_train_data_file}")
log(f"Shared model test parquet path: {shared_model_test_data_file}")
log(f"CatBoost native train parquet path: {catboost_native_train_data_file}")
log(f"CatBoost native test parquet path: {catboost_native_test_data_file}")
log(f"Modeling tables directory: {modeling_tables_dir}")
log(f"Modeling figures directory: {modeling_figures_dir}")


## 1. Modeling Population

In [4]:
# --------------------------------------------------------
# Load feature_base datasets + checkpoint
# --------------------------------------------------------

df_feature_base_train = pd.read_parquet(feature_base_train_data_file)
df_feature_base_test = pd.read_parquet(feature_base_test_data_file)

feature_base_overview_df = da.build_dataset_overview(
    df_train=df_feature_base_train,
    df_test=df_feature_base_test,
)

display(feature_base_overview_df)
log(f"[feature_base] overview: {feature_base_overview_df.to_dict(orient='records')}")

,dataset_split,rows,columns,memory_mb
0,train,466287,66,206.33
1,test,421095,66,186.34


In [5]:
# --------------------------------------------------------
# Inspect loan_status distribution in feature_base
# --------------------------------------------------------

loan_status_overview_df = pd.DataFrame(
    {
        "train": df_feature_base_train["loan_status"].value_counts(dropna=False),
        "test": df_feature_base_test["loan_status"].value_counts(dropna=False),
    }
).fillna(0).astype(int)

display(loan_status_overview_df)

log(f"[feature_base] loan_status distribution inspected")

,train,test
loan_status,,
charged_off,52474,11075
current,173578,346702
default,111,93
does_not_meet_the_credit_policy._status:charged_off,761,0
does_not_meet_the_credit_policy._status:fully_paid,1988,0
fully_paid,227611,48844
in_grace_period,3526,5122
late_(16-30_days),1109,1711
late_(31-120_days),5129,7548


In [6]:
# --------------------------------------------------------
# Build terminal cohort
# --------------------------------------------------------

terminal_statuses = [
    "fully_paid",
    "charged_off",
    "default",
    "does_not_meet_the_credit_policy._status:fully_paid",
    "does_not_meet_the_credit_policy._status:charged_off",
]

df_model_population_train = mp.build_terminal_cohort(
    df=df_feature_base_train,
    status_column="loan_status",
    terminal_statuses=terminal_statuses,
    log=PROJECT_LOG_FILE,
)

df_model_population_test = mp.build_terminal_cohort(
    df=df_feature_base_test,
    status_column="loan_status",
    terminal_statuses=terminal_statuses,
    log=PROJECT_LOG_FILE,
)

{
    "stage": "terminal_cohort_built",
    "train_rows": int(df_model_population_train.shape[0]),
    "test_rows": int(df_model_population_test.shape[0]),
    "train_columns": int(df_model_population_train.shape[1]),
    "test_columns": int(df_model_population_test.shape[1]),
}

{'stage': 'terminal_cohort_built',
 'train_rows': 282945,
 'test_rows': 60012,
 'train_columns': 66,
 'test_columns': 66}

In [7]:
# --------------------------------------------------------
# Modeling population summary
# --------------------------------------------------------

modeling_population_summary_df = ma.build_modeling_population_summary_table(
    df_feature_base_train=df_feature_base_train,
    df_feature_base_test=df_feature_base_test,
    df_model_train=df_model_population_train,
    df_model_test=df_model_population_test,
    train_label="train",
    test_label="test",
    log=PROJECT_LOG_FILE,
)

modeling_population_summary_output_path = modeling_tables_dir / "modeling_population_summary.csv"
modeling_population_summary_df.to_csv(
    modeling_population_summary_output_path,
    index=False,
)


log(f"[model_population] summary written to: {modeling_population_summary_output_path}")


{
    "stage": "modeling_population_summary_created",
    "rows": int(modeling_population_summary_df.shape[0]),
    "output_path": str(modeling_population_summary_output_path),
}

{'stage': 'modeling_population_summary_created',
 'rows': 4,
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\modeling_population_summary.csv'}

In [8]:
# --------------------------------------------------------
# Create binary target
# --------------------------------------------------------

positive_statuses = [
    "charged_off",
    "default",
    "does_not_meet_the_credit_policy._status:charged_off",
]

df_model_population_train = mp.create_target_default(
    df=df_model_population_train,
    status_column="loan_status",
    positive_statuses=positive_statuses,
    target_column="target_default",
    log=PROJECT_LOG_FILE,
)

df_model_population_test = mp.create_target_default(
    df=df_model_population_test,
    status_column="loan_status",
    positive_statuses=positive_statuses,
    target_column="target_default",
    log=PROJECT_LOG_FILE,
)

{
    "stage": "target_created",
    "target_column": "target_default",
    "train_target_non_null": int(df_model_population_train["target_default"].notna().sum()),
    "test_target_non_null": int(df_model_population_test["target_default"].notna().sum()),
}

{'stage': 'target_created',
 'target_column': 'target_default',
 'train_target_non_null': 282945,
 'test_target_non_null': 60012}

In [9]:
# --------------------------------------------------------
# Target distribution artifact
# --------------------------------------------------------

target_distribution_df = ma.build_target_distribution_table(
    df_train=df_model_population_train,
    df_test=df_model_population_test,
    target_column="target_default",
    train_label="train",
    test_label="test",
    log=PROJECT_LOG_FILE,
)

target_distribution_output_path = modeling_tables_dir / "target_distribution.csv"

target_distribution_df.to_csv(
    target_distribution_output_path,
    index=False,
)

log(f"[target_distribution] artifact written to: {target_distribution_output_path}")

{
    "stage": "target_distribution_created",
    "rows": int(target_distribution_df.shape[0]),
    "output_path": str(target_distribution_output_path),
}

{'stage': 'target_distribution_created',
 'rows': 2,
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\target_distribution.csv'}

In [10]:
# --------------------------------------------------------
# Create dataframes for feature engineering
# --------------------------------------------------------

df_feature_engineering_train = df_model_population_train.copy()
df_feature_engineering_test = df_model_population_test.copy()

display(f"'target_default' in train: {'target_default' in df_feature_engineering_train.columns}")
display(f"'target_default' in test: {'target_default' in df_feature_engineering_test.columns}")

"'target_default' in train: True"

"'target_default' in test: True"


## Modeling Population – Conclusion

This stage restricts the dataset to loans with **terminal repayment outcomes** and constructs the binary prediction target `target_default`.

Loans in active repayment states (e.g., current, late, or in grace period) are excluded because their final credit outcome is not yet observed. Policy-related loan statuses indicating that a loan did not meet LendingClub’s internal credit policy are interpreted according to their underlying economic outcomes when constructing the target variable.

The resulting datasets therefore contain only loans with **realized repayment outcomes** and a clearly defined binary target suitable for predictive modeling.

Population reduction and target balance are documented in the modeling artifacts produced in this stage.

To establish a clear pipeline boundary, the resulting datasets are persisted as:

- `model_train_data.parquet`
- `model_test_data.parquet`

These files represent the finalized **modeling population** and serve as the input for the next stage of the pipeline.

---

The analysis now proceeds to **feature engineering**, where the submission-time feature space is prepared for model training.

## 2. Feature Engineering

In [11]:
# --------------------------------------------------------
# Validate modeling datasets
# --------------------------------------------------------

target_column = "target_default"

train_columns = set(df_feature_engineering_train.columns)
test_columns = set(df_feature_engineering_test.columns)

train_only_columns = sorted(train_columns - test_columns)
test_only_columns = sorted(test_columns - train_columns)

if target_column not in df_feature_engineering_train.columns:
    raise KeyError(f"'{target_column}' not found in training data.")

if target_column not in df_feature_engineering_test.columns:
    raise KeyError(f"'{target_column}' not found in testing data.")

validation_checkpoint = {
    "target_column": target_column,
    "train_shape": df_feature_engineering_train.shape,
    "test_shape": df_feature_engineering_test.shape,
    "train_column_count": len(df_feature_engineering_train.columns),
    "test_column_count": len(df_feature_engineering_test.columns),
    "train_only_columns": train_only_columns,
    "test_only_columns": test_only_columns,
}

log(f"[model_dataset_validation] {validation_checkpoint}")

validation_checkpoint

{'target_column': 'target_default',
 'train_shape': (282945, 67),
 'test_shape': (60012, 67),
 'train_column_count': 67,
 'test_column_count': 67,
 'train_only_columns': [],
 'test_only_columns': []}

In [12]:
# --------------------------------------------------------
# Build feature group audit table for modeling datasets
# --------------------------------------------------------

feature_group_audit_df = fe.build_feature_group_audit(
    df_train=df_feature_engineering_train,
    df_test=df_feature_engineering_test,
    target_column=target_column,
    log=PROJECT_LOG_FILE,
)

display(
    feature_group_audit_df.loc[
        feature_group_audit_df["feature_group"] == "categorical",
        ["feature_name", "train_non_null_unique_values", "test_non_null_unique_values"]
    ].sort_values(by="train_non_null_unique_values", ascending=False)
)

display(
    feature_group_audit_df.loc[
        (feature_group_audit_df["feature_group"] == "categorical") &
        (feature_group_audit_df["train_non_null_unique_values"] > 20)
    ]
)

display(
    feature_group_audit_df.loc[
        feature_group_audit_df["is_binary"],
        ["feature_name", "feature_group"]
    ]
)

display(
    feature_group_audit_df.loc[
        feature_group_audit_df["requires_manual_review"]
    ]
)

,feature_name,train_non_null_unique_values,test_non_null_unique_values
2,purpose,14,14
1,loan_status,5,3
0,home_ownership,4,3


,feature_name,train_dtype,test_dtype,feature_group,is_binary,train_non_null_unique_values,test_non_null_unique_values,requires_manual_review


,feature_name,feature_group
15,has_mths_since_last_delinq,numeric
16,has_mths_since_last_major_derog,numeric
17,has_mths_since_last_record,numeric
18,has_mths_since_recent_bc_dlq,numeric
19,has_mths_since_recent_revol_delinq,numeric
57,term_months,numeric


,feature_name,train_dtype,test_dtype,feature_group,is_binary,train_non_null_unique_values,test_non_null_unique_values,requires_manual_review


In [13]:
# --------------------------------------------------------
# Save feature group audit artifact
# --------------------------------------------------------

feature_group_audit_file = modeling_tables_dir / "feature_group_audit.csv"

feature_group_audit_df.to_csv(feature_group_audit_file, index=False)

log(f"[feature_engineering][feature_group_audit] table_file={feature_group_audit_file}")

{
    "feature_group_audit_rows": int(feature_group_audit_df.shape[0]),
    "feature_group_audit_columns": feature_group_audit_df.columns.tolist(),
    "feature_group_audit_file": str(feature_group_audit_file),
}

{'feature_group_audit_rows': 66,
 'feature_group_audit_columns': ['feature_name',
  'train_dtype',
  'test_dtype',
  'feature_group',
  'is_binary',
  'train_non_null_unique_values',
  'test_non_null_unique_values',
  'requires_manual_review'],
 'feature_group_audit_file': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\feature_group_audit.csv'}

In [14]:
# --------------------------------------------------------
# Define feature groups for transformation
# --------------------------------------------------------

columns_to_drop = [
    "loan_status",
    "row_id",
]

categorical_features = [
    "purpose",
    "home_ownership",
]

binary_features = [
    "has_mths_since_last_delinq",
    "has_mths_since_last_major_derog",
    "has_mths_since_last_record",
    "has_mths_since_recent_bc_dlq",
    "has_mths_since_recent_revol_delinq",
]

numeric_features = [
    feature_name
    for feature_name in feature_group_audit_df["feature_name"].tolist()
    if feature_name not in columns_to_drop
    and feature_name not in categorical_features
    and feature_name not in binary_features
]

feature_group_checkpoint = {
    "columns_to_drop_count": len(columns_to_drop),
    "categorical_feature_count": len(categorical_features),
    "binary_feature_count": len(binary_features),
    "numeric_feature_count": len(numeric_features),
}

log(f"[feature_engineering][feature_groups] {feature_group_checkpoint}")

feature_group_checkpoint

{'columns_to_drop_count': 2,
 'categorical_feature_count': 2,
 'binary_feature_count': 5,
 'numeric_feature_count': 57}

In [15]:
# --------------------------------------------------------
# Validate feature group coverage
# --------------------------------------------------------

all_accounted_columns = (
    set(columns_to_drop)
    | set(categorical_features)
    | set(binary_features)
    | set(numeric_features)
    | {target_column}
)

if all_accounted_columns != set(df_feature_engineering_train.columns):
    missing_columns = sorted(set(df_feature_engineering_train.columns) - all_accounted_columns)
    extra_columns = sorted(all_accounted_columns - set(df_feature_engineering_train.columns))
    raise ValueError(
        f"Feature group coverage mismatch. Missing={missing_columns}, Extra={extra_columns}"
    )

feature_group_validation_checkpoint = {
    "feature_group_validation_passed": True,
    "accounted_feature_columns": len(all_accounted_columns),
    "train_total_columns": len(df_feature_engineering_train.columns),
    "missing_columns": [],
    "extra_columns": [],
    "target_column": target_column,
}

log(f"[feature_engineering][feature_group_validation] {feature_group_validation_checkpoint}")

feature_group_validation_checkpoint

{'feature_group_validation_passed': True,
 'accounted_feature_columns': 67,
 'train_total_columns': 67,
 'missing_columns': [],
 'extra_columns': [],
 'target_column': 'target_default'}

In [16]:
#--------------------------------------------------------
# Build numeric skewness summary table
#--------------------------------------------------------

numeric_skewness_summary_df = ma.build_numeric_skewness_summary(
    df_train=df_feature_engineering_train,
    numeric_features=numeric_features,
    log=PROJECT_LOG_FILE,
)

numeric_skewness_summary_file = modeling_tables_dir / "numeric_skewness_summary.csv"

numeric_skewness_summary_df.to_csv(
    numeric_skewness_summary_file,
    index=False,
)

log(f"[feature_engineering][numeric_skewness_summary] table_file={numeric_skewness_summary_file}")

{
    "numeric_skewness_summary_rows": int(numeric_skewness_summary_df.shape[0]),
    "numeric_skewness_summary_columns": numeric_skewness_summary_df.columns.tolist(),
    "numeric_skewness_summary_file": str(numeric_skewness_summary_file),
}

{'numeric_skewness_summary_rows': 57,
 'numeric_skewness_summary_columns': ['feature_name',
  'non_null_count',
  'skewness',
  'min_value',
  'median_value',
  'mean_value',
  'max_value'],
 'numeric_skewness_summary_file': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\numeric_skewness_summary.csv'}

In [17]:
# -------------------------------------------------------------------------------------------
# Show examples of features with high skewness that may benefit from log transformation
# -------------------------------------------------------------------------------------------

report_log_transform_feature_examples = [
    "annual_inc",
    "loan_amnt",
    "revol_bal",
    "tot_cur_bal",
]

log_transformation_comparison_figure_file = (
    modeling_figures_dir / "log_transformation_comparison.png"
)

rf.plot_log_transformation_comparison_figure(
    df_train=df_feature_engineering_train,
    feature_names=report_log_transform_feature_examples,
    output_path=log_transformation_comparison_figure_file,
    log=PROJECT_LOG_FILE,
)

log(
    "[feature_engineering][log_transformation_comparison_figure] "
    f"figure_file={log_transformation_comparison_figure_file}"
)

{
    "log_transformation_comparison_feature_count": len(report_log_transform_feature_examples),
    "log_transformation_comparison_features": report_log_transform_feature_examples,
    "log_transformation_comparison_figure_file": str(log_transformation_comparison_figure_file),
}

{'log_transformation_comparison_feature_count': 4,
 'log_transformation_comparison_features': ['annual_inc',
  'loan_amnt',
  'revol_bal',
  'tot_cur_bal'],
 'log_transformation_comparison_figure_file': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\figures\\log_transformation_comparison.png'}

In [18]:
#------------------------------------------------------------------------------------
# Define feature groups for transformation based on audit and skewness analysis
#------------------------------------------------------------------------------------

columns_to_drop = [
    "loan_status",
    "row_id",
]

categorical_features = [
    "purpose",
    "home_ownership",
]

binary_features = [
    "has_mths_since_last_delinq",
    "has_mths_since_last_major_derog",
    "has_mths_since_last_record",
    "has_mths_since_recent_bc_dlq",
    "has_mths_since_recent_revol_delinq",
]

numeric_features = [
    feature_name
    for feature_name in feature_group_audit_df["feature_name"].tolist()
    if feature_name not in columns_to_drop
    and feature_name not in categorical_features
    and feature_name not in binary_features
]

log_transform_features = [
    "annual_inc",
    "loan_amnt",
    "revol_bal",
    "tot_coll_amt",
    "total_bal_ex_mort",
    "avg_cur_bal",
    "bc_open_to_buy",
    "tot_cur_bal",
    "tot_hi_cred_lim",
    "total_bc_limit",
    "total_il_high_credit_limit",
    "total_rev_hi_lim",
]

## Feature Transformation Strategy

Feature transformation is defined after reviewing feature type, cardinality, and numeric distribution shape in the modeling population.

Two columns will be removed from the modeling feature space.

- `loan_status` will be dropped because it reflects repayment outcome and would leak target information into the model.
- `row_id` will be dropped because it is a structural identifier rather than a borrower attribute.

---

Categorical transformation will be limited to:

- `purpose`
- `home_ownership`

These are the only retained borrower-facing categorical features in the modeling feature space. Both are low-cardinality application-time attributes and can be one-hot encoded without materially expanding dimensionality.

For the models, this keeps categorical handling simple and consistent across the pipeline. Logistic Regression can use these variables without imposing an artificial ordinal structure, while Random Forest and CatBoost will receive the same shared encoded matrix. This preserves comparability across models and avoids introducing model-specific preprocessing branches.

---

The following binary indicators will be retained in their original form:

- `has_mths_since_last_delinq`
- `has_mths_since_last_major_derog`
- `has_mths_since_last_record`
- `has_mths_since_recent_bc_dlq`
- `has_mths_since_recent_revol_delinq`

These variables already represent the presence or absence of specific credit events. Because they are structurally binary, additional transformation would not improve representation.

---

Log transformation will be applied selectively to the following numeric features:

- `annual_inc`
- `loan_amnt`
- `revol_bal`
- `tot_coll_amt`
- `total_bal_ex_mort`
- `avg_cur_bal`
- `bc_open_to_buy`
- `tot_cur_bal`
- `tot_hi_cred_lim`
- `total_bc_limit`
- `total_il_high_credit_limit`
- `total_rev_hi_lim`

These features were selected because they are monetary or exposure-related variables with substantial right skew. In raw form, most observations are concentrated at lower values while a relatively small number of cases extend far into the upper tail. This creates scale dominance: extreme values take up disproportionate space without necessarily carrying proportionally more predictive information. Applying a log transformation compresses that upper tail while preserving rank order, making these variables easier to learn from, especially for the logistic regression baseline.

The remaining numeric features will be left untransformed. This is not because they lack skew, but because skew alone is not treated as sufficient reason to apply a log transform.

1. Count variables such as delinquencies, inquiries, account totals, and public records will be kept in their original form because they represent discrete event frequency rather than continuous monetary scale. Logging them would weaken interpretability while offering limited modeling benefit.

2. Ratio and bounded variables such as `dti`, `revol_util`, `bc_util`, `percent_bc_gt_75`, and `pct_tl_nvr_dlq` will also remain unchanged. These features already live on constrained scales, so the main problem is not extreme magnitude. A log transform would therefore add complexity without solving a meaningful representation issue.

3. Time-since variables will also be excluded from log transformation. Several of these variables use sentinel-coded values for missing historical events, so their skewness reflects data construction as much as borrower behavior. In that setting, a log transform would compress a missingness convention rather than improve feature representation.

4. Lower-range numeric variables such as terms, counts, and simple credit history totals will be retained as-is because their original scale remains directly interpretable and usable across all three models.

---

The resulting policy is selective rather than automatic: drop leakage and identifiers, one-hot encode low-cardinality categoricals, retain binary indicators as-is, log-transform scale-heavy monetary features, and leave counts, ratios, and duration-style variables in their original form.

In [19]:
#--------------------------------------------------------------------------------
# Create branches of modeling datasets for different transformation approaches
#--------------------------------------------------------------------------------

df_catboost_native_train = df_feature_engineering_train.copy()
df_catboost_native_test = df_feature_engineering_test.copy()

df_tree_no_log_train = df_feature_engineering_train.copy()
df_tree_no_log_test = df_feature_engineering_test.copy()

df_shared_model_train = df_feature_engineering_train.copy()
df_shared_model_test = df_feature_engineering_test.copy()


In [20]:
# ------------------------------------------------------------------------
# Drop excluded columns accross all dataset branches with logging
# ------------------------------------------------------------------------

df_catboost_native_train = fu.drop_columns_with_logging(
    df=df_catboost_native_train,
    columns_to_drop=columns_to_drop,
    dataset_name="train",
    log=PROJECT_LOG_FILE,
)

df_catboost_native_test = fu.drop_columns_with_logging(
    df=df_catboost_native_test,
    columns_to_drop=columns_to_drop,
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

df_tree_no_log_train = fu.drop_columns_with_logging(
    df=df_tree_no_log_train,
    columns_to_drop=columns_to_drop,
    dataset_name="train",
    log=PROJECT_LOG_FILE,
)

df_tree_no_log_test = fu.drop_columns_with_logging(
    df=df_tree_no_log_test,
    columns_to_drop=columns_to_drop,
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

df_shared_model_train = fu.drop_columns_with_logging(
    df=df_shared_model_train,
    columns_to_drop=columns_to_drop,
    dataset_name="train",
    log=PROJECT_LOG_FILE,
)

df_shared_model_test = fu.drop_columns_with_logging(
    df=df_shared_model_test,
    columns_to_drop=columns_to_drop,
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

{
    "stage": "branch_column_drop_completed",
    "dropped_column_count": len(columns_to_drop),
    "dropped_columns": columns_to_drop,
    "shared_model_train_shape": df_shared_model_train.shape,
    "shared_model_test_shape": df_shared_model_test.shape,
    "tree_no_log_train_shape": df_tree_no_log_train.shape,
    "tree_no_log_test_shape": df_tree_no_log_test.shape,
    "catboost_native_train_shape": df_catboost_native_train.shape,
    "catboost_native_test_shape": df_catboost_native_test.shape,
}

{'stage': 'branch_column_drop_completed',
 'dropped_column_count': 2,
 'dropped_columns': ['loan_status', 'row_id'],
 'shared_model_train_shape': (282945, 65),
 'shared_model_test_shape': (60012, 65),
 'tree_no_log_train_shape': (282945, 65),
 'tree_no_log_test_shape': (60012, 65),
 'catboost_native_train_shape': (282945, 65),
 'catboost_native_test_shape': (60012, 65)}

In [21]:
# --------------------------------------------------------
# Build category mapping from training data
# --------------------------------------------------------

categorical_features_for_encoding = [
    feature_name
    for feature_name in categorical_features
    if feature_name in df_shared_model_train.columns
]

categorical_feature_mapping = fe.build_category_mapping(
    df_train=df_shared_model_train,
    feature_names=categorical_features_for_encoding,
    log=PROJECT_LOG_FILE,
)

{
    "stage": "category_mapping_built",
    "categorical_feature_count": len(categorical_features),
    "categorical_features": categorical_features,
    "category_sizes": {
        feature_name: len(category_mapping)
        for feature_name, category_mapping in categorical_feature_mapping.items()
    },
}

{'stage': 'category_mapping_built',
 'categorical_feature_count': 2,
 'categorical_features': ['purpose', 'home_ownership'],
 'category_sizes': {'purpose': 14, 'home_ownership': 4}}

In [22]:
# --------------------------------------------------------
# Apply one-hot encoding
# --------------------------------------------------------

df_shared_model_train = fe.apply_one_hot_encoding(
    df=df_shared_model_train,
    feature_names=categorical_features_for_encoding,
    dataset_name="train",
    category_mapping=categorical_feature_mapping,
    log=PROJECT_LOG_FILE,
)

df_shared_model_test = fe.apply_one_hot_encoding(
    df=df_shared_model_test,
    feature_names=categorical_features_for_encoding,
    dataset_name="test",
    category_mapping=categorical_feature_mapping,
    log=PROJECT_LOG_FILE,
)

df_tree_no_log_train = fe.apply_one_hot_encoding(
    df=df_tree_no_log_train,
	feature_names=categorical_features_for_encoding,
	dataset_name="train",
	category_mapping=categorical_feature_mapping,
	log=PROJECT_LOG_FILE,
)

df_tree_no_log_test = fe.apply_one_hot_encoding(
    df=df_tree_no_log_test,
	feature_names=categorical_features_for_encoding,
	dataset_name="test",
	category_mapping=categorical_feature_mapping,
	log=PROJECT_LOG_FILE,
)

{
    "stage": "one_hot_encoding_applied",
    "shared_model_train_shape": df_shared_model_train.shape,
    "shared_model_test_shape": df_shared_model_test.shape,
    "tree_no_log_train_shape": df_tree_no_log_train.shape,
    "tree_no_log_test_shape": df_tree_no_log_test.shape,
    "encoded_feature_count": len(categorical_features),
    "encoded_features": categorical_features,
}

{'stage': 'one_hot_encoding_applied',
 'shared_model_train_shape': (282945, 81),
 'shared_model_test_shape': (60012, 81),
 'tree_no_log_train_shape': (282945, 81),
 'tree_no_log_test_shape': (60012, 81),
 'encoded_feature_count': 2,
 'encoded_features': ['purpose', 'home_ownership']}

In [23]:
# --------------------------------------------------------
# Apply median imputation for shared model comparison
# --------------------------------------------------------

imputation_features = [
    column_name
    for column_name in df_shared_model_train.columns
    if df_shared_model_train[column_name].isna().any()
]

df_shared_model_train, df_shared_model_test = fe.apply_median_imputation(
    df_train=df_shared_model_train,
    df_test=df_shared_model_test,
    feature_names=imputation_features,
    log=PROJECT_LOG_FILE,
)

df_tree_no_log_train, df_tree_no_log_test = fe.apply_median_imputation(
    df_train=df_tree_no_log_train,
	df_test=df_tree_no_log_test,
	feature_names=imputation_features,
	log=PROJECT_LOG_FILE,
)

log(f"[median_imputation_applied][features] {imputation_features}")

{
    "stage": "median_imputation_applied",
    "imputed_feature_count": len(imputation_features),
    "shared_model_train_shape": df_shared_model_train.shape,
    "shared_model_test_shape": df_shared_model_test.shape,
    "tree_no_log_train_shape": df_tree_no_log_train.shape,
    "tree_no_log_test_shape": df_tree_no_log_test.shape,
    "catboost_native_train_shape": df_catboost_native_train.shape,
    "catboost_native_test_shape": df_catboost_native_test.shape,
    "catboost_native_imputation_applied": False,
}

{'stage': 'median_imputation_applied',
 'imputed_feature_count': 48,
 'shared_model_train_shape': (282945, 81),
 'shared_model_test_shape': (60012, 81),
 'tree_no_log_train_shape': (282945, 81),
 'tree_no_log_test_shape': (60012, 81),
 'catboost_native_train_shape': (282945, 65),
 'catboost_native_test_shape': (60012, 65),
 'catboost_native_imputation_applied': False}

In [24]:
# --------------------------------------------------------
# Apply numeric log transformation
# --------------------------------------------------------

df_shared_model_train = fe.apply_numeric_log_transformation(
    df=df_shared_model_train,
    feature_names=log_transform_features,
    dataset_name="train",
    log=PROJECT_LOG_FILE,
)

df_shared_model_test = fe.apply_numeric_log_transformation(
    df=df_shared_model_test,
    feature_names=log_transform_features,
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

{
    "stage": "shared_model_numeric_log_transformation_applied",
    "shared_model_train_shape": df_shared_model_train.shape,
    "shared_model_test_shape": df_shared_model_test.shape,
    "log_transform_feature_count": len(log_transform_features),
    "log_transform_features": log_transform_features,
}

{'stage': 'shared_model_numeric_log_transformation_applied',
 'shared_model_train_shape': (282945, 81),
 'shared_model_test_shape': (60012, 81),
 'log_transform_feature_count': 12,
 'log_transform_features': ['annual_inc',
  'loan_amnt',
  'revol_bal',
  'tot_coll_amt',
  'total_bal_ex_mort',
  'avg_cur_bal',
  'bc_open_to_buy',
  'tot_cur_bal',
  'tot_hi_cred_lim',
  'total_bc_limit',
  'total_il_high_credit_limit',
  'total_rev_hi_lim']}

In [25]:
# --------------------------------------------------------
# Validate engineered datasets
# --------------------------------------------------------

shared_model_train_columns = set(df_shared_model_train.columns)
shared_model_test_columns = set(df_shared_model_test.columns)

tree_no_log_train_columns = set(df_tree_no_log_train.columns)
tree_no_log_test_columns = set(df_tree_no_log_test.columns)

catboost_native_train_columns = set(df_catboost_native_train.columns)
catboost_native_test_columns = set(df_catboost_native_test.columns)

engineered_dataset_checkpoint = {
    "stage": "feature_engineering_complete",
    "shared_model": {
        "train_shape": df_shared_model_train.shape,
        "test_shape": df_shared_model_test.shape,
        "train_column_count": len(df_shared_model_train.columns),
        "test_column_count": len(df_shared_model_test.columns),
        "train_only_columns": sorted(shared_model_train_columns - shared_model_test_columns),
        "test_only_columns": sorted(shared_model_test_columns - shared_model_train_columns),
    },
    "tree_no_log": {
        "train_shape": df_tree_no_log_train.shape,
        "test_shape": df_tree_no_log_test.shape,
        "train_column_count": len(df_tree_no_log_train.columns),
        "test_column_count": len(df_tree_no_log_test.columns),
        "train_only_columns": sorted(tree_no_log_train_columns - tree_no_log_test_columns),
        "test_only_columns": sorted(tree_no_log_test_columns - tree_no_log_train_columns),
    },
    "catboost_native": {
        "train_shape": df_catboost_native_train.shape,
        "test_shape": df_catboost_native_test.shape,
        "train_column_count": len(df_catboost_native_train.columns),
        "test_column_count": len(df_catboost_native_test.columns),
        "train_only_columns": sorted(catboost_native_train_columns - catboost_native_test_columns),
        "test_only_columns": sorted(catboost_native_test_columns - catboost_native_train_columns),
    },
}

log(f"[feature_engineering_complete] {engineered_dataset_checkpoint}")

engineered_dataset_checkpoint

{'stage': 'feature_engineering_complete',
 'shared_model': {'train_shape': (282945, 81),
  'test_shape': (60012, 81),
  'train_column_count': 81,
  'test_column_count': 81,
  'train_only_columns': [],
  'test_only_columns': []},
 'tree_no_log': {'train_shape': (282945, 81),
  'test_shape': (60012, 81),
  'train_column_count': 81,
  'test_column_count': 81,
  'train_only_columns': [],
  'test_only_columns': []},
 'catboost_native': {'train_shape': (282945, 65),
  'test_shape': (60012, 65),
  'train_column_count': 65,
  'test_column_count': 65,
  'train_only_columns': [],
  'test_only_columns': []}}

In [26]:
# --------------------------------------------------------
# Final model columns artifacts
# --------------------------------------------------------

shared_model_columns_df = pd.DataFrame(
    {"feature_name": df_shared_model_train.columns.tolist()}
)
tree_no_log_columns_df = pd.DataFrame(
    {"feature_name": df_tree_no_log_train.columns.tolist()}
)
catboost_native_columns_df = pd.DataFrame(
    {"feature_name": df_catboost_native_train.columns.tolist()}
)

shared_model_columns_file = modeling_tables_dir / "shared_model_columns.csv"
tree_no_log_columns_file = modeling_tables_dir / "tree_no_log_columns.csv"
catboost_native_columns_file = modeling_tables_dir / "catboost_native_columns.csv"

shared_model_columns_df.to_csv(shared_model_columns_file, index=False)
tree_no_log_columns_df.to_csv(tree_no_log_columns_file, index=False)
catboost_native_columns_df.to_csv(catboost_native_columns_file, index=False)

log(
    "[feature_engineering][final_model_columns] "
    f"shared_model_file={shared_model_columns_file} "
    f"tree_no_log_file={tree_no_log_columns_file} "
    f"catboost_native_file={catboost_native_columns_file}"
)

{
    "stage": "final_model_columns_created",
    "shared_model_rows": int(shared_model_columns_df.shape[0]),
    "tree_no_log_rows": int(tree_no_log_columns_df.shape[0]),
    "catboost_native_rows": int(catboost_native_columns_df.shape[0]),
    "shared_model_output_path": str(shared_model_columns_file),
    "tree_no_log_output_path": str(tree_no_log_columns_file),
    "catboost_native_output_path": str(catboost_native_columns_file),
}

{'stage': 'final_model_columns_created',
 'shared_model_rows': 81,
 'tree_no_log_rows': 81,
 'catboost_native_rows': 65,
 'shared_model_output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\shared_model_columns.csv',
 'tree_no_log_output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\tree_no_log_columns.csv',
 'catboost_native_output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\catboost_native_columns.csv'}

In [27]:
# --------------------------------------------------------
# Validate final model branches
# --------------------------------------------------------

final_branch_checkpoint = {
    "shared_model_train_missing_values": int(df_shared_model_train.isna().sum().sum()),
    "shared_model_test_missing_values": int(df_shared_model_test.isna().sum().sum()),
    "tree_no_log_train_missing_values": int(df_tree_no_log_train.isna().sum().sum()),
    "tree_no_log_test_missing_values": int(df_tree_no_log_test.isna().sum().sum()),
    "catboost_native_train_missing_values": int(df_catboost_native_train.isna().sum().sum()),
    "catboost_native_test_missing_values": int(df_catboost_native_test.isna().sum().sum()),
}

log(f"[final_model_branch_validation] {final_branch_checkpoint}")

final_branch_checkpoint

{'shared_model_train_missing_values': 0,
 'shared_model_test_missing_values': 0,
 'tree_no_log_train_missing_values': 0,
 'tree_no_log_test_missing_values': 0,
 'catboost_native_train_missing_values': 2183204,
 'catboost_native_test_missing_values': 14336}

In [28]:
# --------------------------------------------------------
# Save final model datasets
# --------------------------------------------------------

df_shared_model_train.to_parquet(shared_model_train_data_file)
df_shared_model_test.to_parquet(shared_model_test_data_file)

df_tree_no_log_train.to_parquet(tree_no_log_train_data_file)
df_tree_no_log_test.to_parquet(tree_no_log_test_data_file)

df_catboost_native_train.to_parquet(catboost_native_train_data_file)
df_catboost_native_test.to_parquet(catboost_native_test_data_file)

log(f"[model_datasets_saved] shared_model_train_path={shared_model_train_data_file}")
log(f"[model_datasets_saved] shared_model_test_path={shared_model_test_data_file}")
log(f"[model_datasets_saved] tree_no_log_train_path={tree_no_log_train_data_file}")
log(f"[model_datasets_saved] tree_no_log_test_path={tree_no_log_test_data_file}")
log(f"[model_datasets_saved] catboost_native_train_path={catboost_native_train_data_file}")
log(f"[model_datasets_saved] catboost_native_test_path={catboost_native_test_data_file}")

{
    "stage": "model_datasets_saved",
    "shared_model_train_rows": int(df_shared_model_train.shape[0]),
    "shared_model_test_rows": int(df_shared_model_test.shape[0]),
    "shared_model_train_columns": int(df_shared_model_train.shape[1]),
    "shared_model_test_columns": int(df_shared_model_test.shape[1]),
    "tree_no_log_train_rows": int(df_tree_no_log_train.shape[0]),
    "tree_no_log_test_rows": int(df_tree_no_log_test.shape[0]),
    "tree_no_log_train_columns": int(df_tree_no_log_train.shape[1]),
    "tree_no_log_test_columns": int(df_tree_no_log_test.shape[1]),
    "catboost_native_train_rows": int(df_catboost_native_train.shape[0]),
    "catboost_native_test_rows": int(df_catboost_native_test.shape[0]),
    "catboost_native_train_columns": int(df_catboost_native_train.shape[1]),
    "catboost_native_test_columns": int(df_catboost_native_test.shape[1]),
}

{'stage': 'model_datasets_saved',
 'shared_model_train_rows': 282945,
 'shared_model_test_rows': 60012,
 'shared_model_train_columns': 81,
 'shared_model_test_columns': 81,
 'tree_no_log_train_rows': 282945,
 'tree_no_log_test_rows': 60012,
 'tree_no_log_train_columns': 81,
 'tree_no_log_test_columns': 81,
 'catboost_native_train_rows': 282945,
 'catboost_native_test_rows': 60012,
 'catboost_native_train_columns': 65,
 'catboost_native_test_columns': 65}

## Feature Engineering Conclusion

This stage converted the modeling population into a model-ready feature space.

Leakage and identifier columns were removed, selected monetary features were log-transformed, and the retained categorical variables were one-hot encoded using a training-defined category structure applied consistently across both data splits. The resulting training and testing datasets now share the same feature schema and are suitable for direct model estimation. This creates a stable basis for comparing Logistic Regression, Random Forest, and CatBoost under a common preprocessing pipeline.

With feature engineering complete, the next stage is model training and comparative evaluation.

---

## 3. Modeling

With the modeling population defined and feature engineering complete, the next step is to estimate predictive models on the engineered training data and evaluate how well they generalize to the testing period.

The primary comparison uses a shared feature space and common preprocessing pipeline in order to isolate differences in model flexibility under the same input representation. Within that design, three models are trained:

- Logistic Regression as a linear baseline
- Random Forest as a bagging-based non-linear model
- CatBoost as a boosting-based model

This controlled comparison is intended to show not only whether default risk can be predicted from application-time information, but also how much predictive value is gained by moving from a simple baseline to more flexible model classes under a common preprocessing design.

Because CatBoost can handle missing values natively, an additional run is also included in which CatBoost is trained on the non-imputed engineered dataset. This makes it possible to assess whether CatBoost benefits materially from native missing-value handling, rather than inheriting a preprocessing choice introduced primarily to support the baseline models.

Together, these runs distinguish between two related questions: how model classes compare under a shared modeling pipeline, and whether CatBoost gains additional performance when allowed to use one of its native strengths.

---

In [29]:
# --------------------------------------------------------
# Split model inputs and target
# --------------------------------------------------------

target_column = "target_default"

X_shared_model_train = df_shared_model_train.drop(columns=[target_column]).copy()
y_shared_model_train = df_shared_model_train[target_column].copy()

X_shared_model_test = df_shared_model_test.drop(columns=[target_column]).copy()
y_shared_model_test = df_shared_model_test[target_column].copy()

X_tree_no_log_train = df_tree_no_log_train.drop(columns=[target_column]).copy()
y_tree_no_log_train = df_tree_no_log_train[target_column].copy()

X_tree_no_log_test = df_tree_no_log_test.drop(columns=[target_column]).copy()
y_tree_no_log_test = df_tree_no_log_test[target_column].copy()

X_catboost_native_train = df_catboost_native_train.drop(columns=[target_column]).copy()
y_catboost_native_train = df_catboost_native_train[target_column].copy()

X_catboost_native_test = df_catboost_native_test.drop(columns=[target_column]).copy()
y_catboost_native_test = df_catboost_native_test[target_column].copy()

{
    "stage": "model_inputs_split",
    "target_column": target_column,
    "shared_model_X_train_shape": X_shared_model_train.shape,
    "shared_model_X_test_shape": X_shared_model_test.shape,
    "shared_model_y_train_shape": y_shared_model_train.shape,
    "shared_model_y_test_shape": y_shared_model_test.shape,
    "tree_no_log_X_train_shape": X_tree_no_log_train.shape,
    "tree_no_log_X_test_shape": X_tree_no_log_test.shape,
    "tree_no_log_y_train_shape": y_tree_no_log_train.shape,
    "tree_no_log_y_test_shape": y_tree_no_log_test.shape,
    "catboost_native_X_train_shape": X_catboost_native_train.shape,
    "catboost_native_X_test_shape": X_catboost_native_test.shape,
    "catboost_native_y_train_shape": y_catboost_native_train.shape,
    "catboost_native_y_test_shape": y_catboost_native_test.shape,
}

{'stage': 'model_inputs_split',
 'target_column': 'target_default',
 'shared_model_X_train_shape': (282945, 80),
 'shared_model_X_test_shape': (60012, 80),
 'shared_model_y_train_shape': (282945,),
 'shared_model_y_test_shape': (60012,),
 'tree_no_log_X_train_shape': (282945, 80),
 'tree_no_log_X_test_shape': (60012, 80),
 'tree_no_log_y_train_shape': (282945,),
 'tree_no_log_y_test_shape': (60012,),
 'catboost_native_X_train_shape': (282945, 64),
 'catboost_native_X_test_shape': (60012, 64),
 'catboost_native_y_train_shape': (282945,),
 'catboost_native_y_test_shape': (60012,)}

In [30]:
# --------------------------------------------------------
# Identify CatBoost native categorical features
# --------------------------------------------------------

catboost_native_categorical_features = [
    column_name
    for column_name in X_catboost_native_train.columns
    if str(X_catboost_native_train[column_name].dtype) in {"category", "object"}
]

{
    "stage": "catboost_native_categorical_features_identified",
    "categorical_feature_count": len(catboost_native_categorical_features),
    "categorical_features": catboost_native_categorical_features,
}

{'stage': 'catboost_native_categorical_features_identified',
 'categorical_feature_count': 2,
 'categorical_features': ['home_ownership', 'purpose']}

In [31]:
# --------------------------------------------------------
# Train logistic regression model
# --------------------------------------------------------

logistic_regression_model, logistic_regression_training_metadata = tm.train_logistic_regression(
    X_train=X_shared_model_train,
    y_train=y_shared_model_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    max_iter=1000,
    class_weight=None,
    solver="lbfgs",
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {
                "warning_message": logistic_regression_training_metadata["warning_messages"]
            }
        )
    )

{
    "stage": "logistic_regression_trained",
    "training_branch": "shared_model",
    "model_class": type(logistic_regression_model).__name__,
    "train_rows": int(X_shared_model_train.shape[0]),
    "train_columns": int(X_shared_model_train.shape[1]),
    "warning_count": logistic_regression_training_metadata["warning_count"],
    "convergence_warning_count": logistic_regression_training_metadata["convergence_warning_count"],
}

,warning_message
0,ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):\nSTOP: TOTAL NO. OF ITERATIONS REACHED LIMIT\n\nIncrease the number of iterations to improve the convergence (max_iter=1000).\nYou might also want to scale the data as shown in:\n https://scikit-learn.org/stable/modules/preprocessing.html\nPlease also refer to the documentation for alternative solver options:\n https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression


{'stage': 'logistic_regression_trained',
 'training_branch': 'shared_model',
 'model_class': 'LogisticRegression',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 1,
 'convergence_warning_count': 1}

In [32]:
# --------------------------------------------------------
# Scale inputs for Logistic Regression
# --------------------------------------------------------

X_shared_model_train_logistic, X_shared_model_test_logistic, logistic_scaler = tm.scale_logistic_regression_inputs(
    X_train=X_shared_model_train,
    X_test=X_shared_model_test,
    log=PROJECT_LOG_FILE,
)

{
    "stage": "logistic_inputs_scaled",
    "training_branch": "shared_model",
    "X_shared_model_train_logistic_shape": X_shared_model_train_logistic.shape,
    "X_shared_model_test_logistic_shape": X_shared_model_test_logistic.shape,
}

{'stage': 'logistic_inputs_scaled',
 'training_branch': 'shared_model',
 'X_shared_model_train_logistic_shape': (282945, 80),
 'X_shared_model_test_logistic_shape': (60012, 80)}

In [33]:
# --------------------------------------------------------
# Train logistic regression model after scaling inputs
# --------------------------------------------------------

logistic_regression_model, logistic_regression_training_metadata = tm.train_logistic_regression(
    X_train=X_shared_model_train_logistic,
    y_train=y_shared_model_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    max_iter=2000,
    class_weight=None,
    solver="lbfgs",
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {
                "warning_message": logistic_regression_training_metadata["warning_messages"]
            }
        )
    )

{
    "stage": "logistic_regression_trained",
    "training_branch": "shared_model",
    "input_representation": "scaled_shared_model",
    "model_class": type(logistic_regression_model).__name__,
    "train_rows": int(X_shared_model_train_logistic.shape[0]),
    "train_columns": int(X_shared_model_train_logistic.shape[1]),
    "warning_count": logistic_regression_training_metadata["warning_count"],
    "convergence_warning_count": logistic_regression_training_metadata["convergence_warning_count"],
}

,warning_message


{'stage': 'logistic_regression_trained',
 'training_branch': 'shared_model',
 'input_representation': 'scaled_shared_model',
 'model_class': 'LogisticRegression',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 0,
 'convergence_warning_count': 0}

Logistic Regression was trained on a scaled version of the shared imputed feature matrix. This scaling step was applied only to the linear baseline because it supports stable numerical optimization and does not alter the underlying feature set used for model comparison.

In [34]:
# ---------------------------------------------------------------------------------------------
# Train Random Forest model (shared model branch with log transformation and imputation)
# ---------------------------------------------------------------------------------------------

shared_random_forest_model, random_forest_training_metadata = tm.train_random_forest(
    X_train=X_shared_model_train,
    y_train=y_shared_model_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight=None,
    n_jobs=-1,
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {"warning_message": random_forest_training_metadata["warning_messages"]}
        )
    )
    
{
    "stage": "random_forest_trained",
    "model_class": type(shared_random_forest_model).__name__,
    "train_rows": int(X_shared_model_train.shape[0]),
    "train_columns": int(X_shared_model_train.shape[1]),
    "warning_count": random_forest_training_metadata["warning_count"],
    "n_estimators": random_forest_training_metadata["n_estimators"],
    "max_depth": random_forest_training_metadata["max_depth"],
    "max_features": random_forest_training_metadata["max_features"],
}

,warning_message


{'stage': 'random_forest_trained',
 'model_class': 'RandomForestClassifier',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 0,
 'n_estimators': 300,
 'max_depth': None,
 'max_features': 'sqrt'}

In [35]:
# ---------------------------------------------------------------------------------------------
# Train Random Forest model (No-log, but with imputation)
# ---------------------------------------------------------------------------------------------

tree_no_log_random_forest_model, random_forest_training_metadata = tm.train_random_forest(
    X_train=X_tree_no_log_train,
    y_train=y_tree_no_log_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight=None,
    n_jobs=-1,
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {"warning_message": random_forest_training_metadata["warning_messages"]}
        )
    )
    
{
    "stage": "random_forest_trained",
    "model_class": type(tree_no_log_random_forest_model).__name__,
    "train_rows": int(X_tree_no_log_train.shape[0]),
    "train_columns": int(X_tree_no_log_train.shape[1]),
    "warning_count": random_forest_training_metadata["warning_count"],
    "n_estimators": random_forest_training_metadata["n_estimators"],
    "max_depth": random_forest_training_metadata["max_depth"],
    "max_features": random_forest_training_metadata["max_features"],
}

,warning_message


{'stage': 'random_forest_trained',
 'model_class': 'RandomForestClassifier',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 0,
 'n_estimators': 300,
 'max_depth': None,
 'max_features': 'sqrt'}

In [36]:
# -------------------------------------------------------------------------------------------------------------------
# Train CatBoost with shared modeling dataset (with median imputation+ scaling  + One-hot encoding + log transformation))
# -------------------------------------------------------------------------------------------------------------------

catboost_shared_model, catboost_shared_training_metadata = tm.train_catboost(
    X_train=X_shared_model_train,
    y_train=y_shared_model_train,
    log=PROJECT_LOG_FILE,
    random_state=42,
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False,
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {"warning_message": catboost_shared_training_metadata["warning_messages"]}
        )
    )

{
    "stage": "catboost_shared_trained",
    "model_class": type(catboost_shared_model).__name__,
    "train_rows": int(X_shared_model_train.shape[0]),
    "train_columns": int(X_shared_model_train.shape[1]),
    "warning_count": catboost_shared_training_metadata["warning_count"],
    "iterations": catboost_shared_training_metadata["iterations"],
    "learning_rate": catboost_shared_training_metadata["learning_rate"],
    "depth": catboost_shared_training_metadata["depth"],
}

,warning_message


{'stage': 'catboost_shared_trained',
 'model_class': 'CatBoostClassifier',
 'train_rows': 282945,
 'train_columns': 80,
 'warning_count': 0,
 'iterations': 500,
 'learning_rate': 0.05,
 'depth': 6}

In [37]:
# --------------------------------------------------------
# Train CatBoost model with native categorical handling
# --------------------------------------------------------

catboost_native_model, catboost_native_training_metadata = tm.train_catboost(
    X_train=X_catboost_native_train,
    y_train=y_catboost_native_train,
    categorical_feature_names=catboost_native_categorical_features,
    log=PROJECT_LOG_FILE,
    random_state=42,
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False,
)

with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            {"warning_message": catboost_native_training_metadata["warning_messages"]}
        )
    )

{
    "stage": "catboost_native_trained",
    "model_class": type(catboost_native_model).__name__,
    "train_rows": int(X_catboost_native_train.shape[0]),
    "train_columns": int(X_catboost_native_train.shape[1]),
    "warning_count": catboost_native_training_metadata["warning_count"],
    "iterations": catboost_native_training_metadata["iterations"],
    "learning_rate": catboost_native_training_metadata["learning_rate"],
    "depth": catboost_native_training_metadata["depth"],
}

,warning_message


{'stage': 'catboost_native_trained',
 'model_class': 'CatBoostClassifier',
 'train_rows': 282945,
 'train_columns': 64,
 'warning_count': 0,
 'iterations': 500,
 'learning_rate': 0.05,
 'depth': 6}

## Modeling Conclusion

All planned model variants were successfully estimated on the training data.

The final comparison includes three models trained under a shared preprocessing design — Logistic Regression, Random Forest, and CatBoost — together with an explicit **tree-no-log Random Forest experiment** and an additional **CatBoost native** run on the non-one-hot-encoded engineered dataset. This preserves the main controlled comparison while also testing two preprocessing questions directly: whether Random Forest benefits materially from log transformation, and whether CatBoost benefits materially from native categorical handling.

During initial Logistic Regression training, the solver reached the iteration limit on the unscaled shared feature matrix. The Logistic Regression inputs were then standardized to support stable optimization. This adjustment did not change the underlying feature set used in the comparison, but provided the linear baseline with the numerical preparation required for reliable estimation.

The Random Forest comparison showed that performance was effectively unchanged between the shared branch and the tree-no-log branch. This indicates that numeric log transformation did not meaningfully affect Random Forest performance in this setting. The CatBoost comparison showed a small advantage for the native specification over the shared preprocessed version, suggesting that native categorical handling provided a modest improvement, but not a major one.

With model training complete, the next step is to evaluate out-of-sample performance on the 2015 testing period. The evaluation focuses on ranking quality, classification behavior, and probability quality in order to assess how well application-time borrower information captures realized default risk.

## 4. Evaluation

Model training establishes candidate estimators, but does not by itself answer how well they generalize beyond the training window. Evaluation is therefore conducted on the 2015 testing period in order to assess out-of-sample performance under the temporal split defined for the project.

The comparison includes Logistic Regression, Random Forest, and CatBoost under the shared preprocessing design, together with an additional Random Forest run on the tree-no-log branch and an additional CatBoost run using native categorical handling. This makes it possible to compare model classes under a common input structure while also testing two preprocessing questions directly: whether log transformation materially affects Random Forest performance, and whether CatBoost gains materially from native categorical handling relative to the shared preprocessed representation.

Evaluation focuses on three questions. First, how well do the models rank higher-risk borrowers above lower-risk borrowers? Second, how do their classification outcomes behave on the testing period? Third, how informative and stable are the predicted default probabilities? Together, these results determine which model provides the strongest basis for comparison against LendingClub’s grading system in the final stage of the analysis.

---

In [38]:
# --------------------------------------------------------
# Statistical Evaluation Logistic Regression Model
# --------------------------------------------------------

logistic_regression_test_predictions_df = em.generate_model_predictions(
    model=logistic_regression_model,
    X_data=X_shared_model_test_logistic,
    dataset_name="test",
    model_name="logistic_regression",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

logistic_regression_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_shared_model_test,
    prediction_dataframe=logistic_regression_test_predictions_df,
    model_name="logistic_regression",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Statistical Evaluation Random Forest Shared Model
# --------------------------------------------------------

shared_random_forest_test_predictions_df = em.generate_model_predictions(
    model=shared_random_forest_model,
    X_data=X_shared_model_test,
    dataset_name="test",
    model_name="shared_random_forest",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

shared_random_forest_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_shared_model_test,
    prediction_dataframe=shared_random_forest_test_predictions_df,
    model_name="shared_random_forest",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Statistical Evaluation Random Forest Tree_No_Log Model
# --------------------------------------------------------

tree_no_log_random_forest_test_predictions_df = em.generate_model_predictions(
    model=tree_no_log_random_forest_model,
    X_data=X_tree_no_log_test,
    dataset_name="test",
    model_name="tree_no_log_random_forest",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

tree_no_log_random_forest_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_tree_no_log_test,
    prediction_dataframe=tree_no_log_random_forest_test_predictions_df,
    model_name="tree_no_log_random_forest",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Statistical Evaluation CatBoost Shared Model
# --------------------------------------------------------

catboost_shared_test_predictions_df = em.generate_model_predictions(
    model=catboost_shared_model,
    X_data=X_shared_model_test,
    dataset_name="test",
    model_name="catboost_shared",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

catboost_shared_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_shared_model_test,
    prediction_dataframe=catboost_shared_test_predictions_df,
    model_name="catboost_shared",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Statistical Evaluation CatBoost Native Model
# --------------------------------------------------------

catboost_native_test_predictions_df = em.generate_model_predictions(
    model=catboost_native_model,
    X_data=X_catboost_native_test,
    dataset_name="test",
    model_name="catboost_native",
    log=PROJECT_LOG_FILE,
    threshold=0.5,
)

catboost_native_test_metrics_df = em.build_classification_metrics_table(
    y_true=y_catboost_native_test,
    prediction_dataframe=catboost_native_test_predictions_df,
    model_name="catboost_native",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Combine model metrics into single table
# --------------------------------------------------------

model_comparison_metrics_df = em.combine_metrics_tables(
    metrics_tables=[
        logistic_regression_test_metrics_df,
        shared_random_forest_test_metrics_df,
        tree_no_log_random_forest_test_metrics_df,
        catboost_shared_test_metrics_df,
        catboost_native_test_metrics_df,
    ],
    log=PROJECT_LOG_FILE,
)

display(model_comparison_metrics_df)

model_comparison_metrics_file = modeling_tables_dir / "model_comparison_metrics.csv"
model_comparison_metrics_df.to_csv(model_comparison_metrics_file, index=False)

log(f"[evaluation][model_comparison_metrics] table_file={model_comparison_metrics_file}")

{
    "stage": "model_comparison_metrics_created",
    "rows": int(model_comparison_metrics_df.shape[0]),
    "model_count": int(model_comparison_metrics_df["model_name"].nunique()),
    "models_evaluated": sorted(model_comparison_metrics_df["model_name"].unique().tolist()),
    "output_path": str(model_comparison_metrics_file),
}

,model_name,dataset_name,roc_auc,accuracy,precision,recall,f1,brier_score,true_negative,false_positive,false_negative,true_positive
0,logistic_regression,test,0.705828,0.811304,0.469626,0.107987,0.175597,0.140517,47482,1362,9962,1206
1,shared_random_forest,test,0.709380,0.814121,0.506341,0.046472,0.085131,0.141013,48338,506,10649,519
2,tree_no_log_random_forest,test,0.709370,0.814071,0.504883,0.046293,0.084810,0.141010,48337,507,10651,517
3,catboost_shared,test,0.722055,0.812254,0.482747,0.124015,0.197336,0.138251,47360,1484,9783,1385
4,catboost_native,test,0.723023,0.811421,0.474668,0.125000,0.197888,0.138188,47299,1545,9772,1396


{'stage': 'model_comparison_metrics_created',
 'rows': 5,
 'model_count': 5,
 'models_evaluated': ['catboost_native',
  'catboost_shared',
  'logistic_regression',
  'shared_random_forest',
  'tree_no_log_random_forest'],
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\model_comparison_metrics.csv'}

## Evaluation Progress and Next Step

The first stage of evaluation compared model performance on the 2015 testing period using standard classification metrics. This included ROC AUC, accuracy, precision, recall, F1, Brier score, and confusion matrix counts. Together, these results showed how well each model ranked borrower risk, how conservative or aggressive its default classifications were at the default threshold, and how informative its predicted probabilities were.

At this stage, CatBoost produced the strongest overall results. The native CatBoost specification achieved the highest ROC AUC, with the shared CatBoost model performing almost identically just behind it. Logistic Regression remained a credible baseline, while both Random Forest variants were substantially more conservative, generating fewer false positives but also missing a much larger share of realized defaults. The comparison between the shared and tree-no-log Random Forest branches showed that log transformation had no meaningful effect on Random Forest performance in this setting.

These results establish which models perform best in statistical terms, but they do not yet show the full decision meaning of those errors. The confusion matrix counts treat every loan equally, even though a false negative on a small loan and a false negative on a much larger loan do not carry the same practical consequence. In the same way, a model that appears weaker in event counts could still be preferable if the loans it misclassify are economically less important.

The next step therefore extends evaluation from counts to exposure. For each model, the predicted outcomes will be linked back to loan amounts in the testing data so that true positives, true negatives, false positives, and false negatives can also be assessed in monetary terms. This makes it possible to examine not only which model performs best statistically, but also which model performs best when errors are viewed through the size of the loans involved.

---

In [39]:
# -----------------------------------------------------------------------------------------
# Final validation of test dataset rows across all branches before evaluation
# -----------------------------------------------------------------------------------------
{
    "feature_engineering_test_rows": int(df_feature_engineering_test.shape[0]),
    "shared_model_test_rows": int(df_shared_model_test.shape[0]),
    "tree_no_log_test_rows": int(df_tree_no_log_test.shape[0]),
    "catboost_native_test_rows": int(df_catboost_native_test.shape[0]),
}

{'feature_engineering_test_rows': 60012,
 'shared_model_test_rows': 60012,
 'tree_no_log_test_rows': 60012,
 'catboost_native_test_rows': 60012}

In [40]:
# --------------------------------------------------------
# Prepare raw loan amounts for monetary evaluation
# --------------------------------------------------------

loan_amounts_test = df_feature_engineering_test["loan_amnt"].copy()

{
    "stage": "loan_amounts_prepared",
    "rows": int(loan_amounts_test.shape[0]),
    "missing_values": int(loan_amounts_test.isna().sum()),
}

{'stage': 'loan_amounts_prepared', 'rows': 60012, 'missing_values': 0}

In [41]:
# --------------------------------------------------------
# Monetary Evaluation Logistic Regression Model
# --------------------------------------------------------

logistic_regression_outcome_df = em.build_prediction_outcome_table(
    y_true=y_shared_model_test,
    prediction_dataframe=logistic_regression_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="logistic_regression",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

logistic_regression_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=logistic_regression_outcome_df,
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Monetary Evaluation Random Forest Shared Model
# --------------------------------------------------------

shared_random_forest_outcome_df = em.build_prediction_outcome_table(
    y_true=y_shared_model_test,
    prediction_dataframe=shared_random_forest_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="shared_random_forest",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

shared_random_forest_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=shared_random_forest_outcome_df,
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Monetary Evaluation Random Forest Tree_No_Log Model
# --------------------------------------------------------

tree_no_log_random_forest_outcome_df = em.build_prediction_outcome_table(
    y_true=y_tree_no_log_test,
    prediction_dataframe=tree_no_log_random_forest_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="tree_no_log_random_forest",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

tree_no_log_random_forest_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=tree_no_log_random_forest_outcome_df,
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Monetary Evaluation CatBoost Shared Model
# --------------------------------------------------------

catboost_shared_outcome_df = em.build_prediction_outcome_table(
    y_true=y_shared_model_test,
    prediction_dataframe=catboost_shared_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="catboost_shared",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

catboost_shared_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=catboost_shared_outcome_df,
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Monetary Evaluation CatBoost Native Model
# --------------------------------------------------------

catboost_native_outcome_df = em.build_prediction_outcome_table(
    y_true=y_catboost_native_test,
    prediction_dataframe=catboost_native_test_predictions_df,
    loan_amounts=loan_amounts_test,
    model_name="catboost_native",
    dataset_name="test",
    log=PROJECT_LOG_FILE,
)

catboost_native_loan_outcome_summary_df = em.summarize_outcomes_by_loan_amount(
    outcome_dataframe=catboost_native_outcome_df,
    log=PROJECT_LOG_FILE,
)

# --------------------------------------------------------
# Combine loan outcome summaries into single table
# --------------------------------------------------------

loan_outcome_comparison_df = pd.concat(
    [
        logistic_regression_loan_outcome_summary_df,
        shared_random_forest_loan_outcome_summary_df,
        tree_no_log_random_forest_loan_outcome_summary_df,
        catboost_shared_loan_outcome_summary_df,
        catboost_native_loan_outcome_summary_df,
    ],
    axis=0,
    ignore_index=True,
)

loan_outcome_comparison_display_df = loan_outcome_comparison_df.copy()

money_columns = [
    "total_loan_amnt",
    "average_loan_amnt",
    "median_loan_amnt",
]

for column_name in money_columns:
    loan_outcome_comparison_display_df[column_name] = (
        loan_outcome_comparison_display_df[column_name]
        .map(lambda value: f"${value:,.0f}")
    )

display(loan_outcome_comparison_display_df)

loan_outcome_comparison_file = modeling_tables_dir / "loan_outcome_comparison.csv"
loan_outcome_comparison_df.to_csv(loan_outcome_comparison_file, index=False)

log(f"[evaluation][loan_outcome_comparison] table_file={loan_outcome_comparison_file}")

{
    "stage": "loan_outcome_comparison_created",
    "rows": int(loan_outcome_comparison_df.shape[0]),
    "model_count": int(loan_outcome_comparison_df["model_name"].nunique()),
    "output_path": str(loan_outcome_comparison_file),
}

,model_name,dataset_name,outcome_type,row_count,total_loan_amnt,average_loan_amnt,median_loan_amnt
0,logistic_regression,test,false_negative,9962,"$151,714,900","$15,229","$13,600"
1,logistic_regression,test,false_positive,1362,"$24,344,425","$17,874","$16,000"
2,logistic_regression,test,true_negative,47482,"$681,641,800","$14,356","$12,000"
3,logistic_regression,test,true_positive,1206,"$20,861,225","$17,298","$15,650"
4,shared_random_forest,test,false_negative,10649,"$163,442,400","$15,348","$14,000"
5,shared_random_forest,test,false_positive,506,"$9,103,150","$17,990","$16,512"
6,shared_random_forest,test,true_negative,48338,"$696,883,075","$14,417","$12,000"
7,shared_random_forest,test,true_positive,519,"$9,133,725","$17,599","$16,150"
8,tree_no_log_random_forest,test,false_negative,10651,"$163,484,150","$15,349","$14,000"
9,tree_no_log_random_forest,test,false_positive,507,"$9,157,975","$18,063","$16,625"


{'stage': 'loan_outcome_comparison_created',
 'rows': 20,
 'model_count': 5,
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\loan_outcome_comparison.csv'}

## Monetary Evaluation

Monetary evaluation extends the confusion-matrix comparison by linking predicted outcomes in the testing period back to loan amounts. This shows not only how often each model makes different types of classification errors, but also how much loan exposure is attached to those errors.

The results reinforce the earlier statistical comparison. CatBoost produced the strongest monetary outcome, with the native specification performing best overall and the shared CatBoost model following very closely behind. Both CatBoost variants captured more default-linked loan exposure in the true-positive group and left less missed bad-loan exposure in the false-negative group than Logistic Regression and especially Random Forest.

Logistic Regression remained materially stronger than Random Forest in monetary terms. Although it did not match CatBoost, it identified more bad-loan exposure and missed less risky loan volume than either Random Forest variant. The two Random Forest branches again produced almost identical results, indicating that log transformation did not materially affect Random Forest performance in either statistical or monetary terms.

These results strengthen the main modeling conclusion. CatBoost was the strongest model family not only in statistical evaluation, but also in the financial distribution of classification outcomes. At the same time, its stronger detection of bad-loan exposure came with higher false-positive exposure, which means that threshold choice remains central to the eventual decision use of the model.

---

In [42]:
# --------------------------------------------------------
# Build threshold metrics table for CatBoost native model
# --------------------------------------------------------

catboost_native_threshold_metrics_df = ma.build_threshold_metrics_artifact(
    model=catboost_native_model,
    X_data=X_catboost_native_test,
    y_true=y_catboost_native_test,
    model_name="catboost_native",
    dataset_name="test",
    thresholds=[0.20, 0.30, 0.40, 0.50, 0.60],
    log=PROJECT_LOG_FILE,
)

display(catboost_native_threshold_metrics_df)

catboost_native_threshold_metrics_file = (
    modeling_tables_dir / "catboost_native_threshold_metrics.csv"
)

log(f"[evaluation][threshold_metrics_artifact] table_file={catboost_native_threshold_metrics_file}")

{
    "stage": "threshold_metrics_artifact_created",
    "model_name": "catboost_native",
    "dataset_name": "test",
    "rows": int(catboost_native_threshold_metrics_df.shape[0]),
    "threshold_count": int(catboost_native_threshold_metrics_df["threshold"].nunique()),
    "thresholds": sorted(catboost_native_threshold_metrics_df["threshold"].unique().tolist()),
}

,model_name,dataset_name,threshold,roc_auc,accuracy,precision,recall,f1,brier_score,true_negative,false_positive,false_negative,true_positive
0,catboost_native,test,0.2,0.723023,0.626408,0.294040,0.719198,0.417420,0.138188,29560,19284,3136,8032
1,catboost_native,test,0.3,0.723023,0.746617,0.360161,0.465616,0.406155,0.138188,39606,9238,5968,5200
2,catboost_native,test,0.4,0.723023,0.793658,0.414280,0.262894,0.321665,0.138188,44693,4151,8232,2936
3,catboost_native,test,0.5,0.723023,0.811421,0.474668,0.125000,0.197888,0.138188,47299,1545,9772,1396
4,catboost_native,test,0.6,0.723023,0.815870,0.563305,0.047009,0.086777,0.138188,48437,407,10643,525


{'stage': 'threshold_metrics_artifact_created',
 'model_name': 'catboost_native',
 'dataset_name': 'test',
 'rows': 5,
 'threshold_count': 5,
 'thresholds': [0.2, 0.3, 0.4, 0.5, 0.6]}

In [45]:
# ----------------------------------------------------------------
# Build threshold loan outcome table for CatBoost native model
# ----------------------------------------------------------------

catboost_native_threshold_loan_outcomes_df = ma.build_threshold_loan_outcomes_artifact(
    model=catboost_native_model,
    X_data=X_catboost_native_test,
    y_true=y_catboost_native_test,
    loan_amounts=loan_amounts_test,
    model_name="catboost_native",
    dataset_name="test",
    thresholds=[0.20, 0.30, 0.40, 0.50, 0.60],
    log=PROJECT_LOG_FILE,
)

catboost_native_threshold_loan_outcomes_wide_df = ma.reshape_threshold_loan_outcomes_wide(
    catboost_native_threshold_loan_outcomes_df
)

catboost_native_threshold_loan_outcomes_display_df = (
    catboost_native_threshold_loan_outcomes_wide_df.copy()
)

money_columns = [
    column_name
    for column_name in catboost_native_threshold_loan_outcomes_display_df.columns
    if "loan_amnt" in column_name
]

for column_name in money_columns:
    catboost_native_threshold_loan_outcomes_display_df[column_name] = (
        catboost_native_threshold_loan_outcomes_display_df[column_name]
        .map(lambda value: f"${value:,.0f}")
    )

display(catboost_native_threshold_loan_outcomes_display_df)

catboost_native_threshold_loan_outcomes_file = (
    modeling_tables_dir / "catboost_native_threshold_loan_outcomes.csv"
)
catboost_native_threshold_loan_outcomes_df.to_csv(
    catboost_native_threshold_loan_outcomes_file,
    index=False,
)

log(f"[evaluation][threshold_loan_outcomes_artifact] table_file={catboost_native_threshold_loan_outcomes_file}")

{
    "stage": "threshold_loan_outcomes_artifact_saved",
    "model_name": "catboost_native",
    "dataset_name": "test",
    "rows": int(catboost_native_threshold_loan_outcomes_df.shape[0]),
    "output_path": str(catboost_native_threshold_loan_outcomes_file),
}

,model_name,dataset_name,threshold,average_loan_amnt_false_negative,average_loan_amnt_false_positive,average_loan_amnt_true_negative,average_loan_amnt_true_positive,median_loan_amnt_false_negative,median_loan_amnt_false_positive,median_loan_amnt_true_negative,median_loan_amnt_true_positive,row_count_false_negative,row_count_false_positive,row_count_true_negative,row_count_true_positive,total_loan_amnt_false_negative,total_loan_amnt_false_positive,total_loan_amnt_true_negative,total_loan_amnt_true_positive
0,catboost_native,test,0.2,"$13,671","$16,413","$13,176","$16,148","$12,000","$15,000","$11,000","$15,000",3136,19284,29560,8032,"$42,871,600","$316,503,025","$389,483,200","$129,704,525"
1,catboost_native,test,0.3,"$13,971","$17,860","$13,659","$17,154","$12,000","$16,000","$12,000","$15,550",5968,9238,39606,5200,"$83,376,450","$164,993,325","$540,992,900","$89,199,675"
2,catboost_native,test,0.4,"$14,554","$18,619","$14,067","$17,972","$12,000","$17,000","$12,000","$16,150",8232,4151,44693,2936,"$119,810,450","$77,287,150","$628,699,075","$52,765,675"
3,catboost_native,test,0.5,"$15,041","$19,037","$14,304","$18,336","$13,250","$17,875","$12,000","$16,812",9772,1545,47299,1396,"$146,979,700","$29,412,250","$676,573,975","$25,596,425"
4,catboost_native,test,0.6,"$15,297","$18,709","$14,418","$18,611","$14,000","$17,000","$12,000","$17,000",10643,407,48437,525,"$162,805,400","$7,614,500","$698,371,725","$9,770,725"


{'stage': 'threshold_loan_outcomes_artifact_saved',
 'model_name': 'catboost_native',
 'dataset_name': 'test',
 'rows': 20,
 'output_path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\modeling\\tables\\catboost_native_threshold_loan_outcomes.csv'}

In [46]:
loan_amounts_test.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])

count    60012.000000
mean     14639.777878
std       8744.517801
min       1000.000000
25%       8000.000000
50%      12500.000000
75%      20000.000000
90%      28000.000000
95%      34975.000000
max      35000.000000
Name: loan_amnt, dtype: float64

In [ ]:
# --------------------------------------------------------
# Create loan amount bands for testing-period analysis
# --------------------------------------------------------

loan_amount_band_edges = [0, 10000, 20000, float("inf")]
loan_amount_band_labels = ["small", "medium", "large"]

loan_amount_band_test = pd.cut(
    loan_amounts_test,
    bins=loan_amount_band_edges,
    labels=loan_amount_band_labels,
    right=False,
)

{
    "stage": "loan_amount_bands_created",
    "band_counts": loan_amount_band_test.value_counts(dropna=False).to_dict(),
}

In [48]:
loan_amount_band_summary_df = (
    pd.DataFrame(
        {
            "loan_amount_band": loan_amount_band_test,
            "loan_amnt": loan_amounts_test,
        }
    )
    .groupby("loan_amount_band", observed=False)
    .agg(
        row_count=("loan_amnt", "size"),
        min_loan_amnt=("loan_amnt", "min"),
        median_loan_amnt=("loan_amnt", "median"),
        max_loan_amnt=("loan_amnt", "max"),
    )
    .reset_index()
)

display(loan_amount_band_summary_df)

,loan_amount_band,row_count,min_loan_amnt,median_loan_amnt,max_loan_amnt
0,small,19813,1000,6000.0,9975
1,medium,23502,10000,13600.0,19975
2,large,16697,20000,25000.0,35000


In [49]:
# --------------------------------------------------------
# Build row-level threshold outcomes for CatBoost native
# --------------------------------------------------------

catboost_native_thresholds = [0.20, 0.30, 0.40, 0.50, 0.60]

catboost_native_threshold_row_outcomes = []

for threshold_value in catboost_native_thresholds:
    threshold_predictions_df = em.generate_model_predictions(
        model=catboost_native_model,
        X_data=X_catboost_native_test,
        dataset_name="test",
        model_name="catboost_native",
        log=PROJECT_LOG_FILE,
        threshold=threshold_value,
    )

    threshold_outcome_df = em.build_prediction_outcome_table(
        y_true=y_catboost_native_test,
        prediction_dataframe=threshold_predictions_df,
        loan_amounts=loan_amounts_test,
        model_name="catboost_native",
        dataset_name="test",
        log=PROJECT_LOG_FILE,
        threshold=threshold_value,
    ).copy()

    threshold_outcome_df["threshold"] = threshold_value
    threshold_outcome_df["loan_amount_band"] = loan_amount_band_test

    catboost_native_threshold_row_outcomes.append(threshold_outcome_df)

catboost_native_threshold_row_outcomes_df = pd.concat(
    catboost_native_threshold_row_outcomes,
    axis=0,
    ignore_index=False,
)

{
    "stage": "catboost_native_threshold_row_outcomes_created",
    "rows": int(catboost_native_threshold_row_outcomes_df.shape[0]),
    "threshold_count": int(catboost_native_threshold_row_outcomes_df["threshold"].nunique()),
}

{'stage': 'catboost_native_threshold_row_outcomes_created',
 'rows': 300060,
 'threshold_count': 5}

In [50]:
catboost_native_threshold_band_summary_df = (
    catboost_native_threshold_row_outcomes_df
    .groupby(["threshold", "loan_amount_band", "outcome_type"], observed=False)
    .agg(
        row_count=("loan_amnt", "size"),
        total_loan_amnt=("loan_amnt", "sum"),
        average_loan_amnt=("loan_amnt", "mean"),
        median_loan_amnt=("loan_amnt", "median"),
    )
    .reset_index()
    .sort_values(by=["threshold", "loan_amount_band", "outcome_type"])
    .reset_index(drop=True)
)

display(catboost_native_threshold_band_summary_df)

,threshold,loan_amount_band,outcome_type,row_count,total_loan_amnt,average_loan_amnt,median_loan_amnt
0,0.2,small,false_negative,1220,7219050,5917.254098,6000.0
1,0.2,small,false_positive,4676,28262950,6044.257913,6000.0
2,0.2,small,true_negative,12032,69765200,5798.304521,6000.0
3,0.2,small,true_positive,1885,11724075,6219.668435,6250.0
4,0.2,medium,false_negative,1126,15127975,13435.146536,12750.0
5,0.2,medium,false_positive,8071,112667050,13959.490769,14000.0
6,0.2,medium,true_negative,10720,143604925,13395.981810,13000.0
7,0.2,medium,true_positive,3585,50063225,13964.637378,14000.0
8,0.2,large,false_negative,790,20524575,25980.474684,24862.5
9,0.2,large,false_positive,6537,175573025,26858.348631,25025.0


In [51]:
catboost_native_threshold_band_focus_df = (
    catboost_native_threshold_band_summary_df[
        catboost_native_threshold_band_summary_df["outcome_type"].isin(
            ["false_negative", "false_positive", "true_positive"]
        )
    ]
    .copy()
)

catboost_native_threshold_band_focus_display_df = (
    catboost_native_threshold_band_focus_df.copy()
)

money_columns = [
    "total_loan_amnt",
    "average_loan_amnt",
    "median_loan_amnt",
]

for column_name in money_columns:
    catboost_native_threshold_band_focus_display_df[column_name] = (
        catboost_native_threshold_band_focus_display_df[column_name]
        .map(lambda value: f"${value:,.0f}")
    )

display(catboost_native_threshold_band_focus_display_df)

,threshold,loan_amount_band,outcome_type,row_count,total_loan_amnt,average_loan_amnt,median_loan_amnt
0,0.2,small,false_negative,1220,"$7,219,050","$5,917","$6,000"
1,0.2,small,false_positive,4676,"$28,262,950","$6,044","$6,000"
3,0.2,small,true_positive,1885,"$11,724,075","$6,220","$6,250"
4,0.2,medium,false_negative,1126,"$15,127,975","$13,435","$12,750"
5,0.2,medium,false_positive,8071,"$112,667,050","$13,959","$14,000"
7,0.2,medium,true_positive,3585,"$50,063,225","$13,965","$14,000"
8,0.2,large,false_negative,790,"$20,524,575","$25,980","$24,862"
9,0.2,large,false_positive,6537,"$175,573,025","$26,858","$25,025"
11,0.2,large,true_positive,2562,"$67,917,225","$26,509","$25,000"
12,0.3,small,false_negative,2219,"$13,314,100","$6,000","$6,000"


In [52]:
# --------------------------------------------------------
# Build row-level threshold outcomes for CatBoost native
# --------------------------------------------------------

catboost_native_thresholds = [0.20, 0.30, 0.40, 0.50, 0.60]

catboost_native_threshold_row_outcomes = []

for threshold_value in catboost_native_thresholds:
    threshold_predictions_df = em.generate_model_predictions(
        model=catboost_native_model,
        X_data=X_catboost_native_test,
        dataset_name="test",
        model_name="catboost_native",
        log=PROJECT_LOG_FILE,
        threshold=threshold_value,
    )

    threshold_outcome_df = em.build_prediction_outcome_table(
        y_true=y_catboost_native_test,
        prediction_dataframe=threshold_predictions_df,
        loan_amounts=loan_amounts_test,
        model_name="catboost_native",
        dataset_name="test",
        log=PROJECT_LOG_FILE,
        threshold=threshold_value,
    ).copy()

    threshold_outcome_df["threshold"] = threshold_value
    threshold_outcome_df["loan_amount_band"] = loan_amount_band_test

    catboost_native_threshold_row_outcomes.append(threshold_outcome_df)

catboost_native_threshold_row_outcomes_df = pd.concat(
    catboost_native_threshold_row_outcomes,
    axis=0,
    ignore_index=False,
)

{
    "stage": "catboost_native_threshold_row_outcomes_created",
    "rows": int(catboost_native_threshold_row_outcomes_df.shape[0]),
    "threshold_count": int(catboost_native_threshold_row_outcomes_df["threshold"].nunique()),
}

{'stage': 'catboost_native_threshold_row_outcomes_created',
 'rows': 300060,
 'threshold_count': 5}

In [53]:
# --------------------------------------------------------
# Summarize threshold outcomes by loan amount band
# --------------------------------------------------------

catboost_native_threshold_band_summary_df = (
    catboost_native_threshold_row_outcomes_df
    .groupby(["threshold", "loan_amount_band", "outcome_type"], observed=False)
    .agg(
        row_count=("loan_amnt", "size"),
        total_loan_amnt=("loan_amnt", "sum"),
        average_loan_amnt=("loan_amnt", "mean"),
        median_loan_amnt=("loan_amnt", "median"),
    )
    .reset_index()
    .sort_values(by=["threshold", "loan_amount_band", "outcome_type"])
    .reset_index(drop=True)
)

catboost_native_threshold_band_focus_df = (
    catboost_native_threshold_band_summary_df[
        catboost_native_threshold_band_summary_df["outcome_type"].isin(
            ["false_negative", "false_positive", "true_positive"]
        )
    ]
    .copy()
)

catboost_native_threshold_band_focus_display_df = (
    catboost_native_threshold_band_focus_df.copy()
)

money_columns = [
    "total_loan_amnt",
    "average_loan_amnt",
    "median_loan_amnt",
]

for column_name in money_columns:
    catboost_native_threshold_band_focus_display_df[column_name] = (
        catboost_native_threshold_band_focus_display_df[column_name]
        .map(lambda value: f"${value:,.0f}")
    )

display(catboost_native_threshold_band_focus_display_df)

{
    "stage": "catboost_native_threshold_band_summary_created",
    "rows": int(catboost_native_threshold_band_summary_df.shape[0]),
    "focus_rows": int(catboost_native_threshold_band_focus_df.shape[0]),
}

,threshold,loan_amount_band,outcome_type,row_count,total_loan_amnt,average_loan_amnt,median_loan_amnt
0,0.2,small,false_negative,1220,"$7,219,050","$5,917","$6,000"
1,0.2,small,false_positive,4676,"$28,262,950","$6,044","$6,000"
3,0.2,small,true_positive,1885,"$11,724,075","$6,220","$6,250"
4,0.2,medium,false_negative,1126,"$15,127,975","$13,435","$12,750"
5,0.2,medium,false_positive,8071,"$112,667,050","$13,959","$14,000"
7,0.2,medium,true_positive,3585,"$50,063,225","$13,965","$14,000"
8,0.2,large,false_negative,790,"$20,524,575","$25,980","$24,862"
9,0.2,large,false_positive,6537,"$175,573,025","$26,858","$25,025"
11,0.2,large,true_positive,2562,"$67,917,225","$26,509","$25,000"
12,0.3,small,false_negative,2219,"$13,314,100","$6,000","$6,000"


{'stage': 'catboost_native_threshold_band_summary_created',
 'rows': 60,
 'focus_rows': 45}

## Threshold Analysis by Loan Size

Threshold analysis by loan size showed that the main trade-off was concentrated in medium and large loans rather than in small loans alone. Lower thresholds improved default capture across all loan-size bands, but they also increased false-positive exposure, especially among larger loans where the financial consequences of misclassification were greatest.

At the most aggressive threshold of 0.20, the model captured the largest amount of default-linked loan exposure, but did so at the cost of a very large increase in falsely flagged good-loan exposure across all size bands. This pattern was especially pronounced for large loans, where false-positive exposure increased sharply.

At the default 0.50 threshold, the opposite pattern appeared. False-positive exposure was low, but the model missed a substantial amount of risky loan volume, particularly in the medium- and large-loan bands. This confirmed that the default threshold was overly conservative for a use case focused on limiting missed bad-loan exposure.

A threshold of 0.30 provided the strongest overall balance. Relative to 0.50, it materially reduced false-negative exposure across all loan-size bands, while remaining substantially more controlled than the highly aggressive 0.20 threshold. A threshold of 0.40 remained a plausible conservative alternative, especially where limiting false-positive exposure is more important than maximizing default capture.

Taken together, these results support 0.30 as the most decision-useful overall threshold in this analysis. They also suggest that threshold choice matters most for medium and large loans, where classification errors carry the greatest financial weight.